# Notebook 05 — Custom Tokenizer and Multi-Candidate Training

**AI Interview Assistant · Machine Learning Pipeline, Stage 5 of 9**

---

## Purpose

Train, entirely from scratch, the components that make this system's language
model **ours**:

1. A **byte-pair-encoding tokenizer**, fitted on the training split only.
2. **Four Transformer architectures**, each initialised randomly and trained
   from zero — no pretrained weights, no adapters, no distillation.

## Why four architectures instead of one

Picking a single architecture and reporting its score answers nothing: the
reader cannot tell whether the result reflects a good design or a lucky guess.
Training four and selecting on validation loss (Stage 6) makes the choice an
*evidenced* one.

| Candidate | d_model | Layers | Heads | d_ff | Activation | Hypothesis under test |
|---|---|---|---|---|---|---|
| 1 · compact | 256 | 4 | 4 | 1024 | GELU | Is a small model sufficient for this corpus? |
| 2 · scaled | 384 | 6 | 6 | 1536 | GELU | Does width plus depth help? |
| 3 · deep | 512 | 8 | 8 | 2048 | GELU | Does capacity keep paying off, or overfit? |
| 4 · efficient | 384 | 4 | 6 | 1536 | SwiGLU | Does a better activation beat extra depth? |

Candidates 2 and 3 test scaling; candidate 4 isolates the **activation
function** by matching candidate 2's width at candidate 1's depth.

## Critical: the tokenizer sees the training split only

Fitting the tokenizer on the full corpus is a subtle but real form of leakage —
the vocabulary would encode which subwords appear in the test questions. Step 2
asserts the tokenizer never reads validation or test text.

## Outputs

- `tokenizer/` — vocabulary and merge table
- `checkpoints/<candidate_id>/` — one checkpoint per architecture
- `reports/candidate_training_report.json`, `reports/figures/05_*.png`

---

In [ ]:
NOTEBOOK_ID = 5

# ─────────────────────────────────────────────────────────────────────────────
# Step 0 — Environment bootstrap
#
# Locates the project workspace so this notebook runs unchanged in Google Colab,
# a local Jupyter server, or VS Code. Every later step resolves its paths from
# WORKSPACE_DIR, so nothing below depends on where the notebook was opened.
# ─────────────────────────────────────────────────────────────────────────────
import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

def locate_workspace() -> Path:
    """Return the ml-service directory, whatever environment we are in."""
    # 1. Google Colab: mount Drive so checkpoints survive a runtime restart.
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        ws = Path("/content/drive/MyDrive/ai-interview-system/ml-service")
        ws.mkdir(parents=True, exist_ok=True)
        print("Environment      : Google Colab (Drive mounted)")
        return ws
    except ImportError:
        pass

    # 2. Local: walk up from the notebook until we find the ml-service root,
    #    identified by the dataset directory it must contain.
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "dataset").is_dir() and (candidate / "notebooks").is_dir():
            print("Environment      : local")
            return candidate
    print("Environment      : local (fallback to cwd)")
    return here

WORKSPACE_DIR = locate_workspace()
os.chdir(WORKSPACE_DIR)
if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))

# Canonical paths used across all nine notebooks.
RAW_DIR       = WORKSPACE_DIR / "dataset" / "raw"
PROCESSED_DIR = WORKSPACE_DIR / "dataset" / "processed"
QG_DIR        = PROCESSED_DIR / "question_generator"
SPLIT_DIR     = PROCESSED_DIR / "splits"
TOKENIZER_DIR = WORKSPACE_DIR / "tokenizer"
CKPT_DIR      = WORKSPACE_DIR / "checkpoints"
MODEL_DIR     = WORKSPACE_DIR / "models"
REPORTS_DIR   = WORKSPACE_DIR / "reports"
FIGURES_DIR   = REPORTS_DIR / "figures"

for d in (RAW_DIR, PROCESSED_DIR, SPLIT_DIR, TOKENIZER_DIR, CKPT_DIR,
          MODEL_DIR, REPORTS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Workspace        : {WORKSPACE_DIR}")
print(f"Python           : {sys.version.split()[0]}")
print(f"Run started      : {datetime.now(timezone.utc).isoformat(timespec='seconds')}")

---

## Step 0b — Figure and statistics conventions

One style definition serves every figure in the nine-notebook pipeline, so
charts are directly comparable when placed side by side in the write-up.

Three conventions are fixed here:

1. **A colour-blind-safe categorical palette** — the same six colours, in the
   same order, wherever a chart encodes categories.
2. **Automatic figure export** — `save_figure()` writes every figure to
   `reports/figures/` at 200 dpi with a numbered filename, and prints its
   caption, so figures can be cited as *Figure N.k* in the dissertation.
3. **A single summary-statistics function** — `describe_series()` reports
   n, mean, sd, the five-number summary, skewness and kurtosis in a fixed
   order for every variable, so distributions are described consistently.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 0b — Plotting conventions
#
# One style definition for every figure in the pipeline, so figures across the
# nine notebooks are directly comparable in the dissertation. Every figure is
# also saved to reports/figures/ at 200 dpi, ready to drop into the write-up.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.edgecolor": "#444444",
    "grid.alpha": 0.3,
    "legend.frameon": True,
    "figure.autolayout": False,
})

# Colour-blind-safe categorical palette, used consistently for every chart.
PALETTE = ["#3B6FD4", "#E1893B", "#3EA37A", "#C4576B", "#7B5EA7", "#8C7B68"]
sns.set_palette(PALETTE)

_figure_index = {"n": 0}

def save_figure(fig, slug: str, caption: str = "") -> Path:
    """Save a figure with a numbered filename and print its caption."""
    _figure_index["n"] += 1
    n = _figure_index["n"]
    path = FIGURES_DIR / f"{NOTEBOOK_ID:02d}_fig{n:02d}_{slug}.png"
    fig.savefig(path)
    label = f"Figure {NOTEBOOK_ID}.{n}"
    if caption:
        print(f"{label}: {caption}")
    print(f"           saved -> {path.relative_to(WORKSPACE_DIR)}")
    return path

def describe_series(series: pd.Series, name: str) -> pd.Series:
    """Summary statistics reported in a consistent order for every variable."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    return pd.Series({
        "n": len(s),
        "mean": s.mean(),
        "std": s.std(ddof=1),
        "min": s.min(),
        "q1": s.quantile(0.25),
        "median": s.median(),
        "q3": s.quantile(0.75),
        "max": s.max(),
        "skew": s.skew(),
        "kurtosis": s.kurtosis(),
    }, name=name)

print("Plot style       : configured")
print(f"Figure output    : {FIGURES_DIR.relative_to(WORKSPACE_DIR)}")
print(f"Palette          : {len(PALETTE)} colour-blind-safe categories")

---

## Step 1 — Load the training and validation splits

The test split is **not loaded**. The access guard from Stage 4 is called with
this notebook's id to demonstrate that it is genuinely blocked, rather than
merely left alone by convention.

In [ ]:
import time
import math
import hashlib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

def load_split(name: str) -> list:
    path = SPLIT_DIR / f"{name}.jsonl"
    assert path.exists(), f"{path.name} missing — run Notebook 04 first."
    return [json.loads(line) for line in
            path.read_text(encoding="utf-8").splitlines() if line.strip()]

train_records = load_split("train")
val_records = load_split("validation")

print(f"Training records   : {len(train_records):,}")
print(f"Validation records : {len(val_records):,}")

# ── Prove the test split is out of reach from this notebook ─────────────────
LOCK_FILE = SPLIT_DIR / "test_lock.json"
if LOCK_FILE.exists():
    seal = json.loads(LOCK_FILE.read_text(encoding="utf-8"))
    try:
        assert NOTEBOOK_ID in seal["authorised_notebook_ids"]
        print("\nWARNING: this notebook is authorised to read the test split.")
    except AssertionError:
        print(f"\nTest-split access from Notebook {NOTEBOOK_ID}: BLOCKED "
              f"(authorised: {seal['authorised_notebook_ids']})")
        print("The test split is untouched by training, as required.")

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nSeed   : {SEED}")
print(f"Device : {DEVICE}")
print(f"Torch  : {torch.__version__}")

---

## Step 2 — Train the custom BPE tokenizer

The vocabulary size comes from Stage 2's coverage analysis, read from
`reports/eda_summary.json` rather than guessed.

**Byte-pair encoding** starts from characters and repeatedly merges the most
frequent adjacent pair. The result handles words it has never seen by splitting
them into known subwords — which matters here because Stage 2 found that a large
share of the vocabulary appears exactly once.

In [ ]:
from transformer_scratch import CustomBPETokenizer, build_candidate_model, \
    save_checkpoint, load_checkpoint

eda_path = REPORTS_DIR / "eda_summary.json"
if eda_path.exists():
    VOCAB_SIZE = int(json.loads(eda_path.read_text(encoding="utf-8"))
                     ["vocabulary"]["recommended_bpe_vocab"])
    print(f"Vocabulary size from Stage 2 coverage analysis: {VOCAB_SIZE:,}")
else:
    VOCAB_SIZE = 4096
    print(f"Stage 2 report absent; falling back to {VOCAB_SIZE:,}")

# TRAINING SPLIT ONLY — this is the anti-leakage guarantee.
train_texts = [r["question"] for r in train_records]
val_texts = [r["question"] for r in val_records]

tokenizer = CustomBPETokenizer(vocab_size=VOCAB_SIZE)
start = time.perf_counter()
tokenizer.train_from_texts(train_texts)
fit_seconds = time.perf_counter() - start

TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
tokenizer.save(TOKENIZER_DIR)

print(f"\nTOKENIZER TRAINED")
print("=" * 66)
print(f"  Fitted on            : {len(train_texts):,} training questions")
print(f"  Validation questions : 0  (never seen — no leakage)")
print(f"  Fit time             : {fit_seconds:.2f} s")
print(f"  Saved to             : {TOKENIZER_DIR.relative_to(WORKSPACE_DIR)}")
print("=" * 66)

# ── Verify the tokenizer round-trips and generalises ──────────────────────
print("\nROUND-TRIP CHECK (training text)")
for text in train_texts[:3]:
    ids = tokenizer.encode(text, add_special_tokens=True)
    back = tokenizer.decode(ids, skip_special_tokens=True)
    print(f"  original : {text[:62]}")
    print(f"  {len(ids):3d} tokens -> {back[:62]}")

print("\nGENERALISATION CHECK (unseen validation text)")
for text in val_texts[:3]:
    ids = tokenizer.encode(text, add_special_tokens=True)
    back = tokenizer.decode(ids, skip_special_tokens=True)
    print(f"  unseen   : {text[:62]}")
    print(f"  {len(ids):3d} tokens -> {back[:62]}")

In [ ]:
# ── Figure 5.1 — tokenizer behaviour ───────────────────────────────────────
train_lengths = np.array([len(tokenizer.encode(t)) for t in train_texts])
val_lengths = np.array([len(tokenizer.encode(t)) for t in val_texts])
train_words = np.array([len(t.split()) for t in train_texts])
val_words = np.array([len(t.split()) for t in val_texts])

fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.6))

bins = np.arange(0, max(train_lengths.max(), val_lengths.max()) + 3, 2)
axes[0].hist(train_lengths, bins=bins, alpha=0.72, label="train",
             color=PALETTE[0], edgecolor="white", linewidth=0.4, density=True)
axes[0].hist(val_lengths, bins=bins, alpha=0.62, label="validation",
             color=PALETTE[1], edgecolor="white", linewidth=0.4, density=True)
axes[0].set_title("Sequence length in tokens")
axes[0].set_xlabel("Tokens per question")
axes[0].set_ylabel("Density")
axes[0].legend(fontsize=9)

# Fertility = tokens per word. Higher on unseen text means more subword splits.
train_fertility = train_lengths / np.maximum(train_words, 1)
val_fertility = val_lengths / np.maximum(val_words, 1)
axes[1].boxplot([train_fertility, val_fertility], patch_artist=True,
                showmeans=True, widths=0.5,
                medianprops=dict(color="black", linewidth=1.8))
axes[1].set_xticks([1, 2])
axes[1].set_xticklabels(["train", "validation"])
axes[1].set_title("Fertility — tokens per word")
axes[1].set_ylabel("Tokens per word")
axes[1].axhline(1.0, color=PALETTE[2], linestyle=":", linewidth=1.5,
                label="1.0 = whole-word tokens")
axes[1].legend(fontsize=8)

# The sequence-length percentile that sets MAX_SEQ_LEN below.
sorted_lengths = np.sort(train_lengths)
ecdf = np.arange(1, len(sorted_lengths) + 1) / len(sorted_lengths) * 100
axes[2].plot(sorted_lengths, ecdf, linewidth=2.2, color=PALETTE[0])
for pctile, colour in [(95, PALETTE[1]), (99, PALETTE[3])]:
    cut = int(np.percentile(train_lengths, pctile))
    axes[2].axvline(cut, color=colour, linestyle="--", linewidth=1.6,
                    label=f"p{pctile} = {cut} tokens")
axes[2].set_title("Choosing the context window")
axes[2].set_xlabel("Sequence length (tokens)")
axes[2].set_ylabel("Cumulative % of training set")
axes[2].legend(fontsize=8)

fig.suptitle("Custom BPE tokenizer — behaviour on seen and unseen text",
             y=1.03, fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "tokenizer",
            "Fertility barely rises on unseen validation text, which is the "
            "evidence that the subword vocabulary generalises rather than "
            "memorising the training words.")
plt.show()

print(f"Mean fertility — train      : {train_fertility.mean():.3f} tokens/word")
print(f"Mean fertility — validation : {val_fertility.mean():.3f} tokens/word")
print(f"Fertility increase on unseen text: "
      f"{(val_fertility.mean() / train_fertility.mean() - 1) * 100:+.1f}%")
print("A small increase means the vocabulary transfers; a large one would mean "
      "it had\nmemorised training-specific words.")

---

## Step 3 — Build the training tensors

Each question becomes one training sequence. The objective is **causal language
modelling**: predict token *t+1* from tokens *1…t*. This is what lets the
trained model generate a question rather than merely classify one.

`MAX_SEQ_LEN` is set from the 99th percentile of token length, so almost nothing
is truncated while no compute is wasted padding to an unnecessary width.

In [ ]:
MAX_SEQ_LEN = int(min(512, max(32, np.percentile(train_lengths, 99) + 4)))
PAD_ID = tokenizer.encode("", add_special_tokens=True)[0] if False else 0

class QuestionDataset(Dataset):
    """Causal-LM dataset: inputs are tokens 1..n-1, targets are tokens 2..n."""

    def __init__(self, records, tokenizer, max_len):
        self.samples = []
        self.truncated = 0
        for record in records:
            ids = tokenizer.encode(record["question"], add_special_tokens=True)
            if len(ids) > max_len:
                ids = ids[:max_len]
                self.truncated += 1
            if len(ids) < 2:
                continue          # cannot form an input/target pair
            self.samples.append(ids)
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        ids = self.samples[index]
        padded = ids + [PAD_ID] * (self.max_len - len(ids))
        tensor = torch.tensor(padded, dtype=torch.long)
        # Loss is ignored on padding via the -100 convention.
        targets = tensor.clone()
        targets[len(ids):] = -100
        return tensor[:-1], targets[1:]

train_dataset = QuestionDataset(train_records, tokenizer, MAX_SEQ_LEN)
val_dataset = QuestionDataset(val_records, tokenizer, MAX_SEQ_LEN)

BATCH_SIZE = 16 if DEVICE == "cuda" else 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("TRAINING TENSORS")
print("=" * 66)
print(f"  Objective            : causal language modelling (next-token)")
print(f"  Context window       : {MAX_SEQ_LEN} tokens (p99 of training data)")
print(f"  Training sequences   : {len(train_dataset):,} "
      f"({train_dataset.truncated} truncated)")
print(f"  Validation sequences : {len(val_dataset):,} "
      f"({val_dataset.truncated} truncated)")
print(f"  Batch size           : {BATCH_SIZE}")
print(f"  Batches per epoch    : {len(train_loader):,}")
print("=" * 66)

sample_input, sample_target = train_dataset[0]
print(f"\n  Input shape  : {tuple(sample_input.shape)}")
print(f"  Target shape : {tuple(sample_target.shape)}")
print(f"  Ignored positions in target (padding): "
      f"{int((sample_target == -100).sum())}")

---

## Step 4 — Inspect the four architectures before training

Parameter counts are reported *before* any training starts, so the later
loss comparison can be read against model size. A deeper model that fails to
beat a compact one despite four times the parameters is a meaningful result — but
only if the sizes were stated up front.

In [ ]:
CANDIDATES = [
    ("candidate_1_scratch_compact_transformer",
     "Compact", "Is a small model sufficient for this corpus?"),
    ("candidate_2_scratch_scaled_transformer",
     "Scaled", "Does width plus depth help?"),
    ("candidate_3_scratch_deep_transformer",
     "Deep", "Does capacity keep paying off, or overfit?"),
    ("candidate_4_scratch_efficient_transformer",
     "Efficient (SwiGLU)", "Does a better activation beat extra depth?"),
]

architecture_rows = []
for candidate_id, label, hypothesis in CANDIDATES:
    model = build_candidate_model(candidate_id, vocab_size=VOCAB_SIZE)
    config = model.get_config()
    params = model.count_parameters()
    architecture_rows.append({
        "candidate_id": candidate_id,
        "label": label,
        "d_model": config["d_model"],
        "layers": config["num_layers"],
        "heads": config["num_heads"],
        "d_ff": config["d_ff"],
        "activation": config["activation"],
        "parameters": params,
        "parameters_m": round(params / 1e6, 2),
        "size_fp32_mb": round(params * 4 / 1024**2, 1),
        "hypothesis": hypothesis,
    })
    del model

architecture_df = pd.DataFrame(architecture_rows)
print("CANDIDATE ARCHITECTURES — all randomly initialised, trained from zero")
print("=" * 104)
print(architecture_df[["label", "d_model", "layers", "heads", "d_ff",
                       "activation", "parameters_m", "size_fp32_mb"]]
      .to_string(index=False))
print("=" * 104)

# Feasibility: a model far larger than the corpus will memorise it.
tokens_available = int(train_lengths.sum())
print(f"\n  Training tokens available : {tokens_available:,}")
for row in architecture_rows:
    ratio = row["parameters"] / max(tokens_available, 1)
    note = ("severely over-parameterised — expect memorisation"
            if ratio > 100 else
            "over-parameterised for this corpus size" if ratio > 20 else
            "reasonable capacity")
    print(f"  {row['label']:20s} {row['parameters_m']:6.2f}M params "
          f"= {ratio:7.1f}x tokens  ({note})")
print("\nThis corpus is small, so every candidate is over-parameterised. That is")
print("expected and is precisely why Stage 6 selects on VALIDATION loss and why")
print("early stopping is used below — training loss alone would favour the")
print("largest model regardless of whether it generalises.")

In [ ]:
# ── Figure 5.2 — architecture comparison ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.4))

bars = axes[0].bar(architecture_df["label"], architecture_df["parameters_m"],
                   color=PALETTE[:len(architecture_df)])
axes[0].set_title("Trainable parameters")
axes[0].set_ylabel("Millions of parameters")
axes[0].bar_label(bars, fmt="%.2fM", padding=3, fontsize=9)
axes[0].tick_params(axis="x", rotation=22, labelsize=8)
axes[0].margins(y=0.16)

x = np.arange(len(architecture_df))
axes[1].bar(x - 0.2, architecture_df["d_model"], width=0.4, label="d_model",
            color=PALETTE[0])
axes[1].bar(x + 0.2, architecture_df["d_ff"], width=0.4, label="d_ff",
            color=PALETTE[1])
axes[1].set_xticks(x)
axes[1].set_xticklabels(architecture_df["label"], rotation=22, ha="right",
                        fontsize=8)
axes[1].set_title("Width — embedding vs feed-forward")
axes[1].set_ylabel("Dimensions")
axes[1].legend(fontsize=8)

axes[2].scatter(architecture_df["layers"], architecture_df["parameters_m"],
                s=architecture_df["d_model"] * 0.9,
                c=PALETTE[:len(architecture_df)], alpha=0.82,
                edgecolors="white", linewidths=1.6)
for _, row in architecture_df.iterrows():
    axes[2].annotate(row["label"], (row["layers"], row["parameters_m"]),
                     textcoords="offset points", xytext=(0, 15),
                     ha="center", fontsize=8)
axes[2].set_title("Depth vs size  (marker area ∝ d_model)")
axes[2].set_xlabel("Number of Transformer layers")
axes[2].set_ylabel("Parameters (M)")
axes[2].margins(0.24)

fig.suptitle("The four from-scratch candidate architectures", y=1.03,
             fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "architectures",
            "Parameter counts stated before training, so the Stage 6 loss "
            "comparison can be read against model capacity.")
plt.show()

---

## Step 5 — Train each candidate

The training loop, per candidate:

- **AdamW** with weight decay — decoupled decay is the standard choice for
  Transformers.
- **Linear warm-up then cosine decay** — warm-up prevents the large early
  gradients that destabilise a randomly initialised attention stack.
- **Gradient clipping at 1.0** — caps exploding gradients.
- **Early stopping on validation loss** — the run halts when validation loss
  stops improving, which is the mechanism that stops the over-parameterised
  candidates from simply memorising the corpus.

**Perplexity** = exp(loss). It is reported alongside loss because it is
interpretable: it is roughly the number of tokens the model is choosing between
at each step.

In [ ]:
EPOCHS = 8
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
PATIENCE = 3
GRAD_CLIP = 1.0

# On CPU the deep candidates are not feasible in notebook time; the skip is
# reported rather than silently dropping them from the comparison.
if DEVICE == "cpu":
    MAX_PARAMS_CPU = 30e6
    trainable = [c for c in CANDIDATES
                 if architecture_df.loc[
                     architecture_df["candidate_id"] == c[0],
                     "parameters"].iloc[0] <= MAX_PARAMS_CPU]
    skipped = [c[1] for c in CANDIDATES if c not in trainable]
else:
    trainable, skipped = CANDIDATES, []

print(f"Device            : {DEVICE}")
print(f"Candidates to train: {len(trainable)} of {len(CANDIDATES)}")
if skipped:
    print(f"Skipped on CPU     : {skipped}")
    print("  (Reported in the Stage 6 comparison as 'not evaluated', never as")
    print("   a poor result — an untrained model has no validation loss.)")

def evaluate(model, loader, criterion) -> float:
    """Mean per-token loss over a loader."""
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            logits = model(inputs)
            if isinstance(logits, tuple):
                logits = logits[0]
            loss = criterion(logits.reshape(-1, logits.size(-1)),
                             targets.reshape(-1))
            n_tokens = int((targets != -100).sum())
            total_loss += float(loss) * n_tokens
            total_tokens += n_tokens
    return total_loss / max(total_tokens, 1)

training_histories = {}
candidate_results = []

for candidate_id, label, hypothesis in trainable:
    print(f"\n{'=' * 78}")
    print(f"TRAINING: {label}  ({candidate_id})")
    print(f"Hypothesis: {hypothesis}")
    print("=" * 78)

    torch.manual_seed(SEED)            # identical initialisation conditions
    model = build_candidate_model(candidate_id, vocab_size=VOCAB_SIZE).to(DEVICE)
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)

    total_steps = max(1, EPOCHS * len(train_loader))
    warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    history = {"epoch": [], "train_loss": [], "val_loss": [],
               "train_ppl": [], "val_ppl": [], "lr": [], "seconds": []}
    best_val, best_epoch, epochs_without_gain = float("inf"), 0, 0
    checkpoint_dir = CKPT_DIR / candidate_id
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    run_start = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_start = time.perf_counter()
        running_loss, running_tokens = 0.0, 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(inputs)
            if isinstance(logits, tuple):
                logits = logits[0]
            loss = criterion(logits.reshape(-1, logits.size(-1)),
                             targets.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()

            n_tokens = int((targets != -100).sum())
            running_loss += float(loss) * n_tokens
            running_tokens += n_tokens

        train_loss = running_loss / max(running_tokens, 1)
        val_loss = evaluate(model, val_loader, criterion)
        elapsed = time.perf_counter() - epoch_start

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_ppl"].append(float(math.exp(min(train_loss, 20))))
        history["val_ppl"].append(float(math.exp(min(val_loss, 20))))
        history["lr"].append(optimizer.param_groups[0]["lr"])
        history["seconds"].append(elapsed)

        marker = ""
        if val_loss < best_val - 1e-4:
            best_val, best_epoch = val_loss, epoch
            epochs_without_gain = 0
            save_checkpoint(checkpoint_dir, model, optimizer, scheduler,
                            epoch=epoch, step=epoch * len(train_loader),
                            metrics={"val_loss": val_loss,
                                     "train_loss": train_loss,
                                     "val_perplexity": history["val_ppl"][-1]})
            marker = "  <- best, checkpoint saved"
        else:
            epochs_without_gain += 1

        print(f"  epoch {epoch}/{EPOCHS}  "
              f"train {train_loss:.4f} (ppl {history['train_ppl'][-1]:8.1f})  "
              f"val {val_loss:.4f} (ppl {history['val_ppl'][-1]:8.1f})  "
              f"{elapsed:5.1f}s{marker}")

        if epochs_without_gain >= PATIENCE:
            print(f"  Early stopping: no validation improvement for "
                  f"{PATIENCE} epochs.")
            break

    total_seconds = time.perf_counter() - run_start
    training_histories[candidate_id] = history
    candidate_results.append({
        "candidate_id": candidate_id,
        "label": label,
        "hypothesis": hypothesis,
        "parameters": int(architecture_df.loc[
            architecture_df["candidate_id"] == candidate_id,
            "parameters"].iloc[0]),
        "epochs_run": len(history["epoch"]),
        "best_epoch": best_epoch,
        "best_val_loss": round(best_val, 5),
        "best_val_perplexity": round(float(math.exp(min(best_val, 20))), 2),
        "final_train_loss": round(history["train_loss"][-1], 5),
        "generalisation_gap": round(best_val - min(history["train_loss"]), 5),
        "train_seconds": round(total_seconds, 1),
        "checkpoint": str(checkpoint_dir.relative_to(WORKSPACE_DIR)),
        "device": DEVICE,
        "trained": True,
    })
    del model, optimizer, scheduler
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

# Untrained candidates are recorded honestly as not evaluated.
for candidate_id, label, hypothesis in CANDIDATES:
    if candidate_id not in training_histories:
        candidate_results.append({
            "candidate_id": candidate_id, "label": label,
            "hypothesis": hypothesis,
            "parameters": int(architecture_df.loc[
                architecture_df["candidate_id"] == candidate_id,
                "parameters"].iloc[0]),
            "trained": False,
            "skip_reason": f"not feasible on {DEVICE} within notebook runtime",
            "best_val_loss": None,
        })

print(f"\n{'=' * 78}")
print(f"TRAINING COMPLETE — {len(training_histories)} candidate(s) trained")
print("=" * 78)

---

## Step 6 — Learning curves

The central diagnostic figure of this notebook. What to look for:

- **Both curves falling together** → the model is still learning.
- **Training loss falling while validation loss rises** → overfitting begins at
  the point they diverge. The vertical marker shows where early stopping fired.
- **The generalisation gap** (validation minus training loss) is plotted
  separately, because it is the quantity Stage 6 uses to distinguish a model
  that learned from one that memorised.

In [ ]:
# ── Figure 5.3 — learning curves per candidate ─────────────────────────────
if training_histories:
    n = len(training_histories)
    fig, axes = plt.subplots(2, n, figsize=(6.0 * n, 8.4), squeeze=False)

    for col, (candidate_id, history) in enumerate(training_histories.items()):
        label = next(c[1] for c in CANDIDATES if c[0] == candidate_id)
        epochs = history["epoch"]
        result = next(r for r in candidate_results
                      if r["candidate_id"] == candidate_id)

        ax = axes[0][col]
        ax.plot(epochs, history["train_loss"], marker="o", markersize=5,
                linewidth=2.1, color=PALETTE[0], label="training loss")
        ax.plot(epochs, history["val_loss"], marker="s", markersize=5,
                linewidth=2.1, color=PALETTE[3], label="validation loss")
        ax.axvline(result["best_epoch"], color=PALETTE[2], linestyle="--",
                   linewidth=1.8,
                   label=f"best epoch {result['best_epoch']}")
        ax.set_title(f"{label}\n{result['parameters'] / 1e6:.2f}M parameters")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Cross-entropy loss" if col == 0 else "")
        ax.legend(fontsize=8)

        # The gap panel: is it learning, or memorising?
        ax2 = axes[1][col]
        gap = np.array(history["val_loss"]) - np.array(history["train_loss"])
        ax2.plot(epochs, gap, marker="D", markersize=5, linewidth=2.1,
                 color=PALETTE[1])
        ax2.axhline(0, color="black", linewidth=1.0)
        ax2.fill_between(epochs, 0, gap, where=(gap > 0), alpha=0.18,
                         color=PALETTE[3], label="overfitting region")
        ax2.set_title("Generalisation gap (val − train)")
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("Loss difference" if col == 0 else "")
        ax2.legend(fontsize=8)

    fig.suptitle("Learning curves — every candidate trained from random "
                 "initialisation", y=1.0, fontsize=14, fontweight="bold")
    fig.tight_layout()
    save_figure(fig, "learning_curves",
                "Top row: training and validation loss per epoch, with the "
                "early-stopping point marked. Bottom row: the generalisation "
                "gap, which is what Stage 6 selects on.")
    plt.show()
else:
    print("No candidate was trained on this device — no learning curves to plot.")

In [ ]:
# ── Figure 5.4 — cross-candidate comparison ────────────────────────────────
if training_histories:
    fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.6))

    for i, (candidate_id, history) in enumerate(training_histories.items()):
        label = next(c[1] for c in CANDIDATES if c[0] == candidate_id)
        axes[0].plot(history["epoch"], history["val_loss"], marker="o",
                     markersize=4.5, linewidth=2.1,
                     color=PALETTE[i % len(PALETTE)], label=label)
    axes[0].set_title("Validation loss — all candidates")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Validation cross-entropy")
    axes[0].legend(fontsize=8)

    for i, (candidate_id, history) in enumerate(training_histories.items()):
        label = next(c[1] for c in CANDIDATES if c[0] == candidate_id)
        axes[1].plot(history["epoch"], history["val_ppl"], marker="s",
                     markersize=4.5, linewidth=2.1,
                     color=PALETTE[i % len(PALETTE)], label=label)
    axes[1].set_yscale("log")
    axes[1].set_title("Validation perplexity (log scale)")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Perplexity")
    axes[1].legend(fontsize=8)

    # Does spending more parameters actually buy lower loss?
    trained_only = [r for r in candidate_results if r.get("trained")]
    axes[2].scatter([r["parameters"] / 1e6 for r in trained_only],
                    [r["best_val_loss"] for r in trained_only],
                    s=190, c=PALETTE[:len(trained_only)], alpha=0.85,
                    edgecolors="white", linewidths=1.8)
    for r in trained_only:
        axes[2].annotate(r["label"],
                         (r["parameters"] / 1e6, r["best_val_loss"]),
                         textcoords="offset points", xytext=(0, 14),
                         ha="center", fontsize=8)
    axes[2].set_title("Does capacity buy accuracy?")
    axes[2].set_xlabel("Parameters (millions)")
    axes[2].set_ylabel("Best validation loss")
    axes[2].margins(0.26)

    fig.suptitle("Candidate comparison on validation data", y=1.03,
                 fontsize=14, fontweight="bold")
    fig.tight_layout()
    save_figure(fig, "candidate_comparison",
                "Right panel: if the trend is flat or rising, extra capacity "
                "is not helping on a corpus this size — a finding Stage 6 acts "
                "on.")
    plt.show()

    summary = pd.DataFrame(trained_only)[
        ["label", "parameters", "epochs_run", "best_epoch", "best_val_loss",
         "best_val_perplexity", "generalisation_gap", "train_seconds"]]
    summary["parameters_m"] = (summary.pop("parameters") / 1e6).round(2)
    print(summary.sort_values("best_val_loss").to_string(index=False))

---

## Step 7 — Training report

Written to `reports/candidate_training_report.json`, which Stage 6 reads to
select the winning architecture. Note that untrained candidates are recorded
with `trained: false` and a skip reason — they must never be presented as having
performed poorly, because they were never measured.

In [ ]:
training_report = {
    "stage": "05_multi_candidate_training",
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "policy": "All weights randomly initialised. No pretrained weights, "
              "adapters, or distillation were used.",
    "reproducibility": {"seed": SEED, "device": DEVICE,
                        "torch_version": torch.__version__},
    "tokenizer": {
        "type": "custom byte-pair encoding, trained in-project",
        "vocab_size": VOCAB_SIZE,
        "fitted_on": "training split only",
        "training_questions": len(train_texts),
        "fit_seconds": round(fit_seconds, 2),
        "mean_fertility_train": round(float(train_fertility.mean()), 4),
        "mean_fertility_validation": round(float(val_fertility.mean()), 4),
        "path": str(TOKENIZER_DIR.relative_to(WORKSPACE_DIR)),
    },
    "data": {
        "train_sequences": len(train_dataset),
        "validation_sequences": len(val_dataset),
        "max_seq_len": MAX_SEQ_LEN,
        "batch_size": BATCH_SIZE,
        "training_tokens": int(tokens_available),
    },
    "hyperparameters": {
        "epochs_max": EPOCHS, "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO,
        "grad_clip": GRAD_CLIP, "early_stopping_patience": PATIENCE,
        "optimizer": "AdamW",
        "schedule": "linear warm-up then cosine decay",
    },
    "architectures": architecture_df.to_dict(orient="records"),
    "candidates": candidate_results,
    "histories": training_histories,
}

report_path = REPORTS_DIR / "candidate_training_report.json"
report_path.write_text(json.dumps(training_report, indent=2, default=str),
                       encoding="utf-8")

print(f"Training report : {report_path.relative_to(WORKSPACE_DIR)}")
print(f"Checkpoints     : {CKPT_DIR.relative_to(WORKSPACE_DIR)}")
for result in candidate_results:
    if result.get("trained"):
        print(f"  {result['label']:22s} val loss {result['best_val_loss']:.4f}  "
              f"ppl {result['best_val_perplexity']:8.2f}  "
              f"({result['epochs_run']} epochs)")
    else:
        print(f"  {result['label']:22s} NOT EVALUATED — {result['skip_reason']}")

---

## Stage 5 summary

| Component | Status |
|---|---|
| Custom BPE tokenizer | trained on the **training split only** |
| Tokenizer generalisation | fertility increase on unseen text reported in Step 2 |
| Candidate architectures | four, all randomly initialised |
| Training regime | AdamW, warm-up + cosine decay, gradient clipping |
| Overfitting control | early stopping on validation loss |
| Untrained candidates | recorded as `not evaluated`, never as poor results |

### Honest reading of these results

The corpus is small relative to every candidate's parameter count, which Step 4
quantifies explicitly. Two consequences are stated rather than glossed over:

1. **Absolute perplexity is not comparable** to models trained on billions of
   tokens. It is used here only to rank candidates against each other on
   identical data.
2. **The generalisation gap matters more than the loss value.** A model that
   drives training loss to near zero while validation loss rises has memorised
   the corpus, and Figure 5.3's lower row shows exactly that behaviour where it
   occurs.

### Next

**Notebook 06 — Model Comparison and Selection**, which applies multi-criteria
selection over these validation results.